In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, inspect
from IPython.display import display
import json

# 1. COMANDOS PARA RECARGA AUTOMÁTICA
%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 0)           # dejar que use todo el ancho disponible
pd.set_option('display.max_colwidth', None) # Quitar el límite de ancho de las columnas
pd.set_option('display.expand_frame_repr', False) # Para que no "envuelva" la tabla y se mantenga en una sola fila larga

# Descarga de bases de datos:

In [ ]:
def load_journal_to_dict(db_relative_path="../.data/journal.db"):
    """
    Carga todas las tablas de la base de datos en un diccionario de DataFrames.
    Procesa columnas JSON internas si existen.
    """
    engine = create_engine(f"sqlite:///{db_relative_path}")
    inspector = inspect(engine)
    nombres_tablas = inspector.get_table_names()
    
    if not nombres_tablas:
        print("❌ No se encontraron tablas en la base de datos.")
        return {}

    dfs = {}
    
    # Columnas que sabemos que contienen JSON según tu esquema
    json_cols_to_parse = ['emotions', 'behavioral_errors', 'cognitive_patterns']

    for tabla in nombres_tablas:
        print(f"📊 Cargando tabla: '{tabla}'...")
        df = pd.read_sql(f"SELECT * FROM {tabla}", engine)

        # Procesar columnas JSON si la tabla las tiene
        for col in json_cols_to_parse:
            if col in df.columns:
                # Función para parsear JSON de forma segura
                def parse_json(x):
                    if not x or pd.isna(x): return {}
                    try:
                        res = json.loads(x) if isinstance(x, str) else x
                        if isinstance(res, list):
                            return {f"item_{i}": v for i, v in enumerate(res)}
                        if not isinstance(res, dict):
                            return {"value": res}
                        return res
                    except (json.JSONDecodeError, TypeError):
                        return {}

                # Aplanamos el JSON de esa columna específica
                df_json = pd.json_normalize(df[col].apply(parse_json))
                
                # Si el JSON no estaba vacío, añadimos prefijo y combinamos
                if not df_json.empty:
                    df_json.columns = [f"{col}_{subcol}" for subcol in df_json.columns]
                    df = pd.concat([df.drop(columns=[col]), df_json], axis=1)

        dfs[tabla] = df
        print(f"   ✅ {tabla}: {df.shape[0]} filas, {df.shape[1]} columnas.")

    return dfs

In [ ]:
# --- EJECUCIÓN ---
dfs = load_journal_to_dict()

In [ ]:
dfs['efficiency_audit']

In [ ]:
dfs['efficiency_department']

In [ ]:
dfs['tactical_department']

In [ ]:
dfs['tactical_audit']